In [1]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "12345678"
auth=(username, password)
driver = GraphDatabase.driver(uri, auth=(username, password))

In [2]:
def Map(driver):
    query = """
    MATCH (c)
    WHERE c.kind = "Compound"
    OPTIONAL MATCH (c)-[r]->()
    WITH c.name AS Compound, collect(r.metaedge) AS metaedges
    RETURN Compound,
        size([edge IN metaedges WHERE edge IN ['CtD', 'CpD']]) AS disease_count
    ORDER BY disease_count DESC
    """
    with driver.session() as session:
        result = session.run(query)
        return result.data()

In [17]:
def Reduce(map):
    reduced = []
    prev_count = None
    current_total = 0

    for data in map:
        count = data['disease_count']
        if count == prev_count:
            current_total += 1
        else:
            if prev_count is not None:
                reduced.append({'disease_count': prev_count, 'compound_count': current_total})
            prev_count, current_total = count, 1

    if prev_count is not None:
        reduced.append({'disease_count': prev_count, 'compound_count': current_total})

    return reduced[:5]

In [19]:
def MapReduce(driver):
    results = Map(driver)
    reduced_results = Reduce(results)
    return reduced_results

In [21]:
results = MapReduce(driver)
results

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.AggregationSkippedNull} {category: UNRECOGNIZED} {title: The query contains an aggregation function that skips null values.} {description: null value eliminated in set function.} {position: None} for query: '\n    MATCH (c)\n    WHERE c.kind = "Compound"\n    OPTIONAL MATCH (c)-[r]->()\n    WITH c.name AS Compound, collect(r.metaedge) AS metaedges\n    RETURN Compound,\n        size([edge IN metaedges WHERE edge IN [\'CtD\', \'CpD\']]) AS disease_count\n    ORDER BY disease_count DESC\n    '


[{'disease_count': 19, 'compound_count': 1},
 {'disease_count': 17, 'compound_count': 2},
 {'disease_count': 16, 'compound_count': 2},
 {'disease_count': 14, 'compound_count': 2},
 {'disease_count': 13, 'compound_count': 2}]

In [22]:
results = Map(driver)
results

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.AggregationSkippedNull} {category: UNRECOGNIZED} {title: The query contains an aggregation function that skips null values.} {description: null value eliminated in set function.} {position: None} for query: '\n    MATCH (c)\n    WHERE c.kind = "Compound"\n    OPTIONAL MATCH (c)-[r]->()\n    WITH c.name AS Compound, collect(r.metaedge) AS metaedges\n    RETURN Compound,\n        size([edge IN metaedges WHERE edge IN [\'CtD\', \'CpD\']]) AS disease_count\n    ORDER BY disease_count DESC\n    '


[{'Compound': 'Methotrexate', 'disease_count': 19},
 {'Compound': 'Prednisone', 'disease_count': 17},
 {'Compound': 'Doxorubicin', 'disease_count': 17},
 {'Compound': 'Betamethasone', 'disease_count': 16},
 {'Compound': 'Dexamethasone', 'disease_count': 16},
 {'Compound': 'Epirubicin', 'disease_count': 14},
 {'Compound': 'Prednisolone', 'disease_count': 14},
 {'Compound': 'Triamcinolone', 'disease_count': 13},
 {'Compound': 'Methylprednisolone', 'disease_count': 13},
 {'Compound': 'Etoposide', 'disease_count': 11},
 {'Compound': 'Hydrocortisone', 'disease_count': 10},
 {'Compound': 'Dactinomycin', 'disease_count': 10},
 {'Compound': 'Celecoxib', 'disease_count': 9},
 {'Compound': 'Carboplatin', 'disease_count': 9},
 {'Compound': 'Docetaxel', 'disease_count': 9},
 {'Compound': 'Cisplatin', 'disease_count': 8},
 {'Compound': 'Gabapentin', 'disease_count': 8},
 {'Compound': 'Clonazepam', 'disease_count': 8},
 {'Compound': 'Vincristine', 'disease_count': 7},
 {'Compound': 'Fluorouracil', '